# screamingface · YAML quickstart

Keep a fusion lineup in a small, reviewable YAML file, load it without making model calls, and
run the same URL4-backed evaluation flow as the main quickstart.

This checked-in execution is an explicit **SIMULATION**. It uses the bundled synthetic science
fixture and deterministic local model answers, so it runs offline and makes no provider claims.
For a live run, replace the setup call with `sf.setup()` and update `fusion.yaml` with exact model
IDs returned by that live session's `sf.models.list()`, which lists models from your actively
connected providers. The live prerequisites — a running AI Gateway, connected providers, and
Hugging Face access for gated GPQA — are listed in
[`00_quickstart.ipynb`](00_quickstart.ipynb).

Model IDs are not aliases: `hf/...` is not silently expanded to `huggingface/...`, and an
`open_router/...` model is valid only when the connected gateway reports that exact ID.

## 1 · Start a reproducible session

In [1]:
import screamingface as sf

session = sf.setup(mode="mock", static_widgets=True)
session

SetupPanel(state='connected', credentials=<never stored>)

## 2 · Check the exact model IDs

In [2]:
available = sf.models.list()
available

['codex/gpt-5.5', 'gemini-cli/gemini-2.5-pro', 'anthropic/claude-sonnet-4-6']

## 3 · Review `fusion.yaml`

The YAML file sits beside this notebook. Loading it is local and safe: it does not execute the
fusion, contact AI Gateway, or reveal the URL4 recipe.

```yaml
name: yaml-trio
models:
  - codex/gpt-5.5
  - gemini-cli/gemini-2.5-pro
  - anthropic/claude-sonnet-4-6
reduce: majority_vote
judge: codex/gpt-5.5
```

## 4 · Load the fusion and inspect its lineup

In [3]:
fusion = sf.Fusion.from_yaml("fusion.yaml")
fusion  # rich lineup table; the URL4 stays hidden

Role,Model
Judge,codex/gpt-5.5
Member,gemini-cli/gemini-2.5-pro
Member,anthropic/claude-sonnet-4-6


Loading and inspecting a fusion does not require connected providers. In live mode,
`evaluate(...)` checks every required provider and model before loading benchmark data. If anything
is missing, it raises one `FusionNotReady` error and makes no model calls.

Ask for the canonical, shareable URL4 only when you need it:

In [4]:
fusion.url4

"(sf-model://codex/gpt-5.5, sf-model://gemini-cli/gemini-2.5-pro, sf-model://anthropic/claude-sonnet-4-6)!'majority_vote';sf_version=1;sf_name=yaml-trio;sf_judge=codex/gpt-5.5"

The same schema can be supplied as an inline Python mapping:

In [5]:
fusion_config = {
    "name": "yaml-trio",
    "models": available[:3],
    "reduce": "majority_vote",
    "judge": available[0],
}
inline_fusion = sf.Fusion(**fusion_config)
inline_fusion.url4 == fusion.url4

True

## 5 · Evaluate and read the result

In [6]:
run = fusion.evaluate("gpqa", first=20, seed=0)
run

Run(benchmark='GPQA-shaped synthetic science fixture', dataset_source='synthetic-gpqa-shaped', mode='mock', models=('codex/gpt-5.5', 'gemini-cli/gemini-2.5-pro', 'anthropic/claude-sonnet-4-6'), url="(sf-model://codex/gpt-5.5, sf-model://gemini-cli/gemini-2.5-pro, sf-model://anthropic/claude-sonnet-4-6)!'majority_vote';sf_version=1;sf_name=yaml-trio;sf_judge=codex/gpt-5.5", sample_size=20, seed=0, score=100.0, baseline=80.0, gain=20.0, cost_usd=0.0, fusion_name='yaml-trio', reduce='majority_vote', judge='codex/gpt-5.5', incomplete=0, profiles=(), pricing_source='estimate:SDK catalog', pricing_as_of='2026-07-16', prompt_tokens=0, completion_tokens=0, total_tokens=0, model_results=(ModelResult(model='codex/gpt-5.5', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0), ModelResult(model='gemini-cli/gemini-2.5-pro', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0), ModelResult(model='anthropic/claude-sonnet-4-6', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0)), failures=())

The result card is explicitly simulated. For both simulated and live runs, read
`gain` first: positive means the fusion beat its strongest member using the same panel answers.

In [7]:
{
    "score": run.score,
    "baseline": run.baseline,
    "gain": run.gain,
    "mode": run.mode,
}

{'score': 100.0, 'baseline': 80.0, 'gain': 20.0, 'mode': 'mock'}

**Next:** [`00_quickstart.ipynb`](00_quickstart.ipynb) walks the same flow with an
inline Python lineup and explains how to switch this notebook to a live run.